In [1]:
import math
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from datetime import datetime
from tqdm import tqdm
from torch.optim import LBFGS
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset 
import torch.optim as optim
from utility import create_data_loaders, train_model
from sklearn.preprocessing import LabelEncoder, label_binarize

################################################################################################################
class NaiveFourierKANLayer(nn.Module):
    def __init__(self, inputdim, outdim, gridsize, addbias=True, smooth_initialization=False):
        super(NaiveFourierKANLayer, self).__init__()
        self.gridsize = gridsize
        self.addbias = addbias
        self.inputdim = inputdim
        self.outdim = outdim

        grid_norm_factor = (torch.arange(gridsize) + 1) ** 2 if smooth_initialization else np.sqrt(gridsize)

        self.fouriercoeffs = nn.Parameter(torch.randn(2, outdim, inputdim, gridsize) /
                                          (np.sqrt(inputdim) * grid_norm_factor))
        if self.addbias:
            self.bias = nn.Parameter(torch.zeros(1, outdim))

    def forward(self, x):
        xshp = x.shape
        outshape = xshp[0:-1] + (self.outdim,)
        x = torch.reshape(x, (-1, self.inputdim))
        k = torch.reshape(torch.arange(1, self.gridsize + 1, device=x.device), (1, 1, 1, self.gridsize))
        xrshp = torch.reshape(x, (x.shape[0], 1, x.shape[1], 1))
        c = torch.cos(k * xrshp)
        s = torch.sin(k * xrshp)
        y = torch.sum(c * self.fouriercoeffs[0:1], (-2, -1))
        y += torch.sum(s * self.fouriercoeffs[1:2], (-2, -1))
        if self.addbias:
            y += self.bias
        y = torch.reshape(y, outshape)
        return y

################################################################################################################
class fKAN(nn.Module):
    def __init__(self, layers, gridsizes):
        super(fKAN, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 2):
            self.layers.append(NaiveFourierKANLayer(layers[i], layers[i + 1], gridsize=gridsizes[i]))
        self.layers.append(nn.Linear(layers[len(layers) - 2], layers[len(layers) - 1]))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

################################################################################################################
# --- Mixer Layer with KAN for Token and Channel Mixing ---------------
class MixerLayerKAN(nn.Module):
    def __init__(self, num_tokens, token_dim, channel_dim, token_order=3, channel_order=3):
        super(MixerLayerKAN, self).__init__()
        self.norm1 = nn.LayerNorm(channel_dim)
        self.token_mixing = fKAN(layers=[num_tokens, token_dim, num_tokens], gridsizes=[token_order])
        
        self.norm2 = nn.LayerNorm(channel_dim)
        self.channel_mixing = fKAN(layers=[channel_dim, channel_dim * 2, channel_dim], gridsizes=[channel_order])
        
    def forward(self, x):

        y = self.norm1(x)
        # Transpose to have tokens as features: shape becomes (batch, channel_dim, num_tokens)
        y = y.transpose(1, 2)
        # Flatten batch and channel dims so each "channel" is processed separately: (batch * channel_dim, num_tokens)
        B, C, T = y.shape
        y = y.reshape(B * C, T)
        # Apply token mixing fKAN
        y = self.token_mixing(y)
        # Reshape back and transpose to original dimensions: (batch, num_tokens, channel_dim)
        y = y.reshape(B, C, T).transpose(1, 2)
        # Add residual connection
        x = x + y

        # ----- Channel Mixing -----
        y = self.norm2(x)
        # Now process each token's channel vector: (batch, num_tokens, channel_dim) -> (batch * num_tokens, channel_dim)
        B, T, C = y.shape
        y = y.reshape(B * T, C)
        y = self.channel_mixing(y)
        y = y.reshape(B, T, C)
        x = x + y
        return x

# --- MLPMixer Model with KAN for Tabular Data -----------------------
class KANMixer(nn.Module):
    def __init__(self, num_features, num_classes, num_layers=4, token_dim=64, channel_dim=128,
                 token_order=3, channel_order=3):
        """
        num_features: Number of tokens (features/columns)
        num_classes: Number of target classes
        num_layers: Number of Mixer layers
        token_dim: Hidden size for token mixing
        channel_dim: Embedding dimension for each token
        token_order, channel_order: Degree for Chebyshev polynomial in token and channel mixing respectively
        """
        super(KANMixer, self).__init__()
        self.num_tokens = num_features
        # Embed each feature (scalar) into a vector of size channel_dim.
        self.embedding = nn.Linear(1, channel_dim)
        
        # Sequence of Mixer layers using KAN blocks.
        self.mixer_layers = nn.Sequential(*[
            MixerLayerKAN(num_tokens=self.num_tokens, token_dim=token_dim, channel_dim=channel_dim,
                          token_order=token_order, channel_order=channel_order)
            for _ in range(num_layers)
        ])
        
        # Final normalization and classification head.
        self.norm = nn.LayerNorm(channel_dim)
        self.fc = nn.Linear(channel_dim, num_classes)
        
    def forward(self, x):
     
        x = x.unsqueeze(-1) 
        x = self.embedding(x)  
        x = self.mixer_layers(x)  
        x = self.norm(x)
        x = x.mean(dim=1)  
        logits = self.fc(x) 

        return logits


/home/defuser/miniconda3/envs/KAN/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
################################################################################################################
def objective(trial):
    
    num_layers= trial.suggest_int("num_layers", 1, 4)
    token_dim = trial.suggest_int("token_dim", 32, 128, step=16)
    channel_dim = trial.suggest_int("channel_dim", 32, 128, step=16)
    token_order = trial.suggest_int("token_order", 1, 5)
    channel_order = trial.suggest_int("channel_order", 1, 5)
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
   
    model = KANMixer(
        num_features=input_shape, 
        num_classes=output_shape, 
        num_layers=num_layers,
        token_dim=token_dim,
        channel_dim=channel_dim,
        token_order=token_order,
        channel_order=channel_order
    ).double() 
      
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_loader, valid_loader = create_data_loaders(X_train, y_train, X_valid, y_valid, batch_size=32)
    train_model(model, train_loader, valid_loader, criterion, optimizer, device, num_epochs=EPOCHs)
    
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_valid)
    y_true_bin = label_binarize(y_valid.cpu().numpy(), classes=np.arange(NUM_CLASS))
    auc_score = roc_auc_score(y_true_bin, test_outputs.cpu(), multi_class='ovr')

    return auc_score
################################################################################################################


## Fix your dataset here

In [ ]:
## Modify this part based on your dataset and features

NUM_CLASS = 3

file_path = '/path/to/your/Data.csv'
data = pd.read_csv(file_path)


features = ['feature1',	'column2',	'Header3']   ## Feature columns
target = 'Target label'  # Targer column

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Preprocessing
data[features] = data[features].apply(pd.to_numeric, errors='coerce')
data[features] = data[features].fillna(data[features].mean())
data = data.dropna(subset=[target])
label_encoder = LabelEncoder()
data[target] = label_encoder.fit_transform(data[target])
data[target] = data[target].astype(int)
X = data[features].astype(float).values
y = data[target].values

# Train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, stratify=y_temp, test_size=0.5, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert data to tensors and move to GPU if available
scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float64).to(device)
X_valid = torch.tensor(scaler.transform(X_val), dtype=torch.float64).to(device)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float64).to(device)

y_test_main = y_test.copy()

y_train = torch.nn.functional.one_hot(torch.tensor(y_train), num_classes=NUM_CLASS).to(device).float()
y_valid = torch.nn.functional.one_hot(torch.tensor(y_val), num_classes=NUM_CLASS).to(device).float()
y_test = torch.nn.functional.one_hot(torch.tensor(y_test), num_classes=NUM_CLASS).to(device).float()

input_shape = X_train.shape[1]
output_shape = y_train.shape[1]

if output_shape > 1:
    y_train = y_train.argmax(dim=1)
    y_valid = y_valid.argmax(dim=1)
    y_test = y_test.argmax(dim=1)
################################################################################################################
TRIALS = 100
EPOCHs = 50


####  If you want to optimize your network based on your own dataset, uncomment the following lines:
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=TRIALS)
# best_params = study.best_params

### If you have the parameters from previous optimization, you can directly use them here:
best_params = {'num_layers': 3, 'token_dim': 112, 'channel_dim': 96, 'token_order': 1, 'channel_order': 3, 'lr': 0.0033110417736098158}

# best_params = {
#     "num_layers": 4,
#     "token_dim": 64,
#     "channel_dim" : 64,
#     "token_order": 3,
#     "channel_order": 3,
#     "lr": 0.001
# }

num_layers = best_params["num_layers"]
token_dim = best_params["token_dim"]
channel_dim = best_params["channel_dim"]
token_order = best_params["token_order"]
channel_order = best_params["channel_order"]
lr = best_params["lr"]

model = KANMixer(
    num_features=input_shape, 
    num_classes=output_shape, 
    num_layers=num_layers, 
    token_dim=token_dim, 
    channel_dim=channel_dim, 
    token_order=token_order, 
    channel_order=channel_order
).double()

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
train_loader, valid_loader = create_data_loaders(X_train, y_train, X_valid, y_valid, batch_size=32)
train_model(model, train_loader, valid_loader, criterion, optimizer, device, num_epochs=EPOCHs)

################################################################################################################
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)

# Multiclass ROC AUC
y_true_bin = label_binarize(y_test_main, classes=np.arange(NUM_CLASS))
auc = roc_auc_score(y_true_bin, test_outputs.cpu(), multi_class='ovr')

print('best_params:', best_params)
print("Test ROC AUC:", auc)

################################

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    log_loss,
    cohen_kappa_score,
    matthews_corrcoef,
    top_k_accuracy_score
)

# Probabilistic predictions
with torch.no_grad():
    y_pred_proba = test_outputs.cpu().numpy()
    y_pred = y_pred_proba.argmax(axis=1)

# Calculate metrics
y_test_cpu = y_test_main  # Use the original y_test_main for true labels

auc = roc_auc_score(y_true_bin, y_pred_proba, multi_class='ovr')
accuracy = accuracy_score(y_test_cpu, y_pred)

precision_macro = precision_score(y_test_cpu, y_pred, average='macro')
recall_macro = recall_score(y_test_cpu, y_pred, average='macro')
f1_macro = f1_score(y_test_cpu, y_pred, average='macro')

precision_weighted = precision_score(y_test_cpu, y_pred, average='weighted')
recall_weighted = recall_score(y_test_cpu, y_pred, average='weighted')
f1_weighted = f1_score(y_test_cpu, y_pred, average='weighted')

logloss = log_loss(y_test_cpu, y_pred_proba)

kappa = cohen_kappa_score(y_test_cpu, y_pred)
mcc = matthews_corrcoef(y_test_cpu, y_pred)
top3_accuracy = top_k_accuracy_score(y_test_cpu, y_pred_proba, k=3)
conf_matrix = confusion_matrix(y_test_cpu, y_pred)
report = classification_report(y_test_cpu, y_pred, zero_division=0)

# Print metrics
print(f"AUC: {auc}")
print(f"Accuracy: {accuracy}")
print(f"Precision (Macro): {precision_macro}")
print(f"Recall (Macro): {recall_macro}")
print(f"F1 Score (Macro): {f1_macro}")
print(f"Precision (Weighted): {precision_weighted}")
print(f"Recall (Weighted): {recall_weighted}")
print(f"F1 Score (Weighted): {f1_weighted}")
print(f"Log Loss: {logloss}")
print(f"Cohen's Kappa: {kappa}")
print(f"Matthews Correlation Coefficient: {mcc}")
print(f"Top-3 Accuracy: {top3_accuracy}")
print(f"Confusion Matrix:\n{conf_matrix}")
print(f"Classification Report:\n{report}")

Epoch 1/50 - Train Loss: 0.8171, Valid Loss: 0.7116, Valid Acc: 0.6512
Epoch 2/50 - Train Loss: 0.7580, Valid Loss: 0.7200, Valid Acc: 0.7035
Epoch 3/50 - Train Loss: 0.7389, Valid Loss: 0.7059, Valid Acc: 0.6860
Epoch 4/50 - Train Loss: 0.7334, Valid Loss: 0.7808, Valid Acc: 0.6512
Epoch 5/50 - Train Loss: 0.7596, Valid Loss: 0.7151, Valid Acc: 0.6802
Epoch 6/50 - Train Loss: 0.7352, Valid Loss: 0.7206, Valid Acc: 0.6919
Epoch 7/50 - Train Loss: 0.7176, Valid Loss: 0.7356, Valid Acc: 0.6977
Epoch 8/50 - Train Loss: 0.7112, Valid Loss: 0.7498, Valid Acc: 0.6686
Epoch 9/50 - Train Loss: 0.7157, Valid Loss: 0.7194, Valid Acc: 0.6686
Epoch 10/50 - Train Loss: 0.7103, Valid Loss: 0.7665, Valid Acc: 0.6628
Epoch 11/50 - Train Loss: 0.7302, Valid Loss: 0.7148, Valid Acc: 0.6860
Epoch 12/50 - Train Loss: 0.7035, Valid Loss: 0.7342, Valid Acc: 0.6744
Epoch 13/50 - Train Loss: 0.7014, Valid Loss: 0.7600, Valid Acc: 0.6628
Epoch 14/50 - Train Loss: 0.7031, Valid Loss: 0.7302, Valid Acc: 0.6744
E

/home/defuser/miniconda3/envs/KAN/lib/python3.9/site-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/home/defuser/miniconda3/envs/KAN/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:2022: UndefinedMetricWarning: 'k' (3) greater than or equal to 'n_classes' (3) will result in a perfect score and is therefore meaningless.
  warnings.warn(
